# 27 — COP, SHR, ECOP, eCDD across the full Temperature × Relative-Humidity sweep

Four metrics from `cedar.metrics`, each swept across the **entire** plausible
range of outdoor dry-bulb temperature and relative humidity, shown as a heatmap.
The point is to see *what each metric depends on* — and, in particular, where the
humidity axis matters and where it doesn't:

| metric | call | depends on |
|---|---|---|
| **COP** — coefficient of performance | `SingleFluidCOP.cop(T)` | temperature only |
| **SHR** — sensible heat ratio | `SensibleHeatRatioModel.shr(T, rh_array=RH)` | temperature **and** RH |
| **ECOP** — effective COP = COP × SHR | `EffectiveCOP.ecop(T, rh_array=RH)` | temperature **and** RH |
| **eCDD** — effective cooling degree = CDD / (COP × SHR) | `EffectiveCDD.ecdd(T, rh_array=RH)` | temperature **and** RH |

The headline: **COP is humidity-blind** (its heatmap is vertical bands — the
compressor doesn't know about the air's moisture). The moment you fold in **SHR**
— the fraction of the cooling that is *sensible* rather than spent wringing water
out of the air — the picture tilts, and **eCDD** (the real cooling burden a humid
climate imposes) blows up in the hot-and-humid corner. That corner is Jakarta.


## 1. Setup — import the four `cedar.metrics` models

In [ ]:
import sys, pathlib
import numpy as np
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# This notebook lives in the climload repo; put its src/ layout first on the path
# so `cedar` resolves to climload's metrics package (not the Energyplus cedar).
HERE = pathlib.Path.cwd().resolve()
REPO = HERE.parent if HERE.name == "notebooks" else HERE
sys.path.insert(0, str(REPO / "src"))

from cedar.metrics.cop import SingleFluidCOP
from cedar.metrics.shr import SensibleHeatRatioModel
from cedar.metrics.cdd import CoolingDegreeDays
from cedar.metrics.effective_cop import EffectiveCOP
from cedar.metrics.effective_cdd import EffectiveCDD

print("cedar.metrics loaded from:", pathlib.Path(SingleFluidCOP.__module__.replace('.', '/')))


## 2. Build the models

Each model is constructed once at a fixed system design point, then evaluated
across the whole climate grid.

- **COP** — a single-refrigerant vapour-compression model (R134a). The
  evaporator is fixed at **0 °C** (a comfort cooling coil, matching the SHR
  model's coil) and the condenser tracks **ambient + 10 K**, so the lift grows —
  and COP falls — as the outdoor air gets hotter.
- **SHR** — the default comfort-cooling coil (saturated air leaving a ~0 °C coil).
  SHR is where relative humidity enters: humid air carries a large latent load, so
  a smaller fraction of the coil's work is sensible cooling.
- **CDD** — cooling degree exceedance above an 18 °C base.
- **ECOP** = COP × SHR and **eCDD** = CDD / (COP × SHR) compose the three.


In [ ]:
COP_FLUID      = "R134a"
T_EVAP_K       = 0.0 + 273.15    # evaporator fixed at 0 °C (comfort cooling coil)
DELTA_T_COND   = 10.0            # condenser = ambient + 10 K
DELTA_T_MIN    = 5.0             # minimum compressor lift [K]
CDD_BASE_K     = 18.0 + 273.15   # cooling-degree base temperature

# Condenser is ambient + 10 K against a 0 °C evaporator, so the lift is
# (T_amb + 10) − 0. Below T_amb ≈ −5 °C that lift drops under DELTA_T_MIN and COP
# is undefined (you would free-cool, not run the compressor); those cells come
# back NaN and render blank — the physically honest picture.
cop_model  = SingleFluidCOP(COP_FLUID, t_evap_k=T_EVAP_K, delta_t_cond=DELTA_T_COND,
                            delta_t_min=DELTA_T_MIN)
shr_model  = SensibleHeatRatioModel()                         # default comfort coil
cdd_model  = CoolingDegreeDays(base_temperature_K=CDD_BASE_K)
ecop_model = EffectiveCOP(cop_model=cop_model, shr_model=shr_model)
ecdd_model = EffectiveCDD(cop_model=cop_model, shr_model=shr_model, cdd_model=cdd_model)

print("models ready:")
print(f"  COP  : {COP_FLUID}, evap {T_EVAP_K-273.15:.0f} C, cond approach {DELTA_T_COND:.0f} K")
print(f"  SHR  : default comfort coil (saturated exit)")
print(f"  CDD  : base {CDD_BASE_K-273.15:.0f} C")


## 3. The full Temperature × Relative-Humidity sweep grid

We sweep outdoor dry-bulb from a cold **−10 °C** up to a brutal **48 °C**, and
relative humidity across its entire physical range **0 → 100 %**. Every metric is
evaluated on this same grid so the four heatmaps are directly comparable.

The models take temperature in **Kelvin** and RH as a **fraction (0–1)**; we keep
Celsius / percent for the axes.


In [ ]:
T_C   = np.linspace(-10.0, 48.0, 240)     # outdoor dry-bulb [C]
RH_PC = np.linspace(0.0, 100.0, 200)      # relative humidity [%]

# Meshgrid: rows = RH, cols = T, so imshow puts T on x and RH on y.
TT_C, RR_PC = np.meshgrid(T_C, RH_PC)
TT_K  = TT_C + 273.15
RR_FR = np.clip(RR_PC / 100.0, 1e-6, 1.0)  # fraction, avoid exactly 0 for SHR

# Evaluate the four metrics on the grid.
cop_grid  = np.asarray(cop_model.cop(TT_K), dtype=float)
shr_grid  = np.asarray(shr_model.shr(TT_K, rh_array=RR_FR), dtype=float)
ecop_grid = np.asarray(ecop_model.ecop(TT_K, rh_array=RR_FR), dtype=float)
ecdd_grid = np.asarray(ecdd_model.ecdd(TT_K, rh_array=RR_FR), dtype=float)

# eCDD is only meaningful where there is a cooling load (T above the CDD base);
# below the base CDD = 0 so eCDD = 0. Mask the no-load region for a clean colour scale.
ecdd_masked = np.where(TT_C > (CDD_BASE_K - 273.15), ecdd_grid, np.nan)

for name, g in [("COP", cop_grid), ("SHR", shr_grid),
                ("ECOP", ecop_grid), ("eCDD", ecdd_grid)]:
    finite = g[np.isfinite(g)]
    print(f"{name:5s} range: {finite.min():.3f} .. {finite.max():.3f}")


## 4. Four heatmaps — the full sweep

Read them together, left→right, top→bottom:

1. **COP** — perfectly vertical bands. The compressor cares only about how hot it
   must reject heat to; RH is invisible to it.
2. **SHR** — now the humidity axis bites. Low RH → SHR near 1 (almost all sensible
   cooling). High RH → SHR collapses as the coil spends its capacity condensing
   water instead of dropping temperature.
3. **ECOP = COP × SHR** — the thermodynamic ceiling (COP) *discounted* by the
   latent penalty (SHR). The best corner is cool-and-dry; the worst is hot-and-humid.
4. **eCDD = CDD / (COP × SHR)** — the effective cooling burden. It stays zero below
   the 18 °C base (no cooling load), then climbs with temperature and, crucially,
   **explodes toward high RH**: the same hot day costs far more cooling when the air
   is wet. This is the map that explains why a Jakarta (hot **and** humid) is a
   fundamentally harder site than a Phoenix (hot but dry).


In [ ]:
extent = [T_C.min(), T_C.max(), RH_PC.min(), RH_PC.max()]

def show(ax, grid, title, cmap, *, norm=None, cbar_label=""):
    im = ax.imshow(grid, origin="lower", extent=extent, aspect="auto",
                   cmap=cmap, norm=norm)
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_xlabel("outdoor dry-bulb temperature [°C]")
    ax.set_ylabel("relative humidity [%]")
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(cbar_label)
    return im

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor("#fafafa")

show(axes[0, 0], cop_grid, "COP — temperature only (humidity-blind)",
     "viridis", cbar_label="COP [W/W]")
show(axes[0, 1], shr_grid, "SHR — sensible fraction (drops in humid air)",
     "cividis", cbar_label="SHR [-]")
show(axes[1, 0], ecop_grid, "ECOP = COP × SHR — efficiency after the latent penalty",
     "magma", cbar_label="effective COP [W/W]")

# eCDD spans orders of magnitude → log colour scale over the cooling-load region.
finite_ecdd = ecdd_masked[np.isfinite(ecdd_masked) & (ecdd_masked > 0)]
norm = LogNorm(vmin=max(finite_ecdd.min(), 1e-3), vmax=finite_ecdd.max())
im = show(axes[1, 1], ecdd_masked,
          "eCDD = CDD / (COP × SHR) — real cooling burden (log scale)",
          "inferno", norm=norm, cbar_label="effective CDD [K] (log)")
axes[1, 1].text(0.03, 0.08,
                "blank = below 18 °C base\n(no cooling load)",
                transform=axes[1, 1].transAxes, fontsize=8, color="#333",
                bbox=dict(boxstyle="round", fc="white", ec="#999", alpha=0.9))

fig.suptitle("Cooling metrics across the full Temperature × Relative-Humidity sweep",
             fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


## 5. The one-line takeaway, quantified

Compare two hot days at the same **35 °C** dry-bulb — one bone-dry, one tropical —
to put a number on how much humidity alone changes the cooling burden.


In [ ]:
def metrics_at(t_c, rh_pc):
    t_k = np.array([t_c + 273.15])
    rh = np.array([rh_pc / 100.0])
    return {
        "COP":  float(cop_model.cop(t_k)[0]),
        "SHR":  float(shr_model.shr(t_k, rh_array=rh)[0]),
        "ECOP": float(ecop_model.ecop(t_k, rh_array=rh)[0]),
        "eCDD": float(ecdd_model.ecdd(t_k, rh_array=rh)[0]),
    }

dry    = metrics_at(35.0, 20.0)     # 35 C, 20% RH — Phoenix-like
humid  = metrics_at(35.0, 85.0)     # 35 C, 85% RH — Jakarta-like

print("Same 35 °C dry-bulb, two humidities:")
print(f"{'metric':6s} {'dry 20% RH':>12s} {'humid 85% RH':>14s} {'ratio':>8s}")
for k in ["COP", "SHR", "ECOP", "eCDD"]:
    r = humid[k] / dry[k] if dry[k] else float('nan')
    print(f"{k:6s} {dry[k]:12.3f} {humid[k]:14.3f} {r:7.2f}x")

print()
print("COP is identical (temperature-only). But the humid day's SHR collapses, so")
print(f"its effective cooling burden (eCDD) is ~{humid['eCDD']/dry['eCDD']:.1f}x the dry day's —")
print("at the SAME temperature. Humidity, not heat, is what makes the tropics hard.")


## 6. A real year on the map — Jakarta, hour by hour

The sweeps above are the *possible* space. Now drop a **real city** onto it. We load
the Jakarta ERA5 weather year (lesson 26's `weather_apac/` library), compute the
**effective COP for every one of the 8,760 hours** — `ECOP = COP(T) × SHR(T, dewpoint)`
— and see where a whole tropical year actually lands.

The raw CSV stores dry-bulb and dewpoint in **Kelvin**, which is exactly what the
metric models want (`shr(..., dewpoint_array=...)`), so no manual humidity
conversion is needed.


In [ ]:
import pandas as pd

# Jakarta from the lesson-26 weather library (prefer 2025, fall back to 2023).
WEATHER_APAC = pathlib.Path.cwd().parent / "notebooks" / "weather_apac" \
    if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd() / "notebooks" / "weather_apac"
candidates = sorted(WEATHER_APAC.glob("era5_jakarta_*_raw.csv"), reverse=True)
assert candidates, f"no Jakarta CSV in {WEATHER_APAC}; run lesson 26 first"
JAKARTA_CSV = candidates[0]
year = JAKARTA_CSV.stem.split("_")[2]
print("loading", JAKARTA_CSV.name)

jak = pd.read_csv(JAKARTA_CSV)
jak["utc_time"] = pd.to_datetime(jak["utc_time"], utc=True)
T_k  = jak["2m_temperature"].to_numpy()           # Kelvin
Td_k = jak["2m_dewpoint_temperature"].to_numpy()  # Kelvin
T_c  = T_k - 273.15

# Per-hour metrics from the SAME models built in §2.
jak_cop  = np.asarray(cop_model.cop(T_k), dtype=float)
jak_shr  = np.asarray(shr_model.shr(T_k, dewpoint_array=Td_k), dtype=float)
jak_ecop = np.asarray(ecop_model.ecop(T_k, dewpoint_array=Td_k), dtype=float)
# RH (%) just for the plot axis.
from cedar.physics.shr import compute_relative_humidity
jak_rh = np.asarray(compute_relative_humidity(T_k, Td_k), dtype=float) * 100.0

print(f"Jakarta {year}: {len(jak):,} hours")
print(f"  dry-bulb  {T_c.min():5.1f} .. {T_c.max():5.1f} C  (mean {T_c.mean():.1f})")
print(f"  RH        {jak_rh.min():5.0f} .. {jak_rh.max():5.0f} %  (mean {jak_rh.mean():.0f})")
print(f"  eCOP      {np.nanmin(jak_ecop):5.2f} .. {np.nanmax(jak_ecop):5.2f}    (mean {np.nanmean(jak_ecop):.2f})")


## 7. The plot, built up in three passes

You asked to *see how a plot is made*, so we build the same axes three ways and stack
them, each pass adding one idea:

1. **Connected trajectory** — every hour is a dot, joined to the next in time order.
   Over a year this traces the city's actual path through temperature–humidity space.
   It is busy on purpose: that tangle *is* the climate.
2. **Soft pastel cloud** — the same points as a translucent density blob, so the eye
   reads *where the year mostly sat* instead of chasing the thread.
3. **Labelled summary** — the cloud plus a few annotations (the year's mean point, the
   hottest hour, the muggiest hour) so the picture explains itself.

All three sit on top of the eCOP field from §3–4, so you can read each Jakarta hour's
effective COP straight off the background colour.


In [ ]:
from matplotlib.colors import Normalize

fig, axes = plt.subplots(1, 3, figsize=(19, 5.6))
fig.patch.set_facecolor("#fafafa")

# Shared background: the eCOP field (ECOP grid from §3), faint, as context.
def ecop_backdrop(ax):
    ax.imshow(ecop_grid, origin="lower",
              extent=[T_C.min(), T_C.max(), RH_PC.min(), RH_PC.max()],
              aspect="auto", cmap="magma", alpha=0.35, vmin=0)
    ax.set_xlim(20, 40); ax.set_ylim(40, 100)   # zoom to the tropical corner
    ax.set_xlabel("dry-bulb [°C]"); ax.set_ylabel("relative humidity [%]")

# ---- Pass 1: connected trajectory, coloured by time of year ----------------
ax = axes[0]; ecop_backdrop(ax)
order = np.argsort(jak["utc_time"].values)
ax.plot(T_c[order], jak_rh[order], "-", lw=0.25, color="#3a3a3a", alpha=0.5, zorder=2)
sc = ax.scatter(T_c, jak_rh, c=jak["utc_time"].dt.dayofyear, cmap="twilight",
                s=6, alpha=0.8, zorder=3)
ax.set_title("1 — every hour, connected in time\n(colour = day of year)", fontsize=10, fontweight="bold")
plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label="day of year")

# ---- Pass 2: soft pastel cloud ---------------------------------------------
ax = axes[1]; ecop_backdrop(ax)
# Layered translucent scatter = a soft density cloud (no extra deps).
for size, alpha in [(90, 0.015), (45, 0.02), (18, 0.03)]:
    ax.scatter(T_c, jak_rh, s=size, color="#8ecae6", alpha=alpha,
               edgecolors="none", zorder=3)
ax.set_title("2 — the same hours as a soft cloud\n(where the year actually sat)",
             fontsize=10, fontweight="bold")

# ---- Pass 3: labelled summary ----------------------------------------------
ax = axes[2]; ecop_backdrop(ax)
for size, alpha in [(90, 0.015), (45, 0.02), (18, 0.03)]:
    ax.scatter(T_c, jak_rh, s=size, color="#8ecae6", alpha=alpha,
               edgecolors="none", zorder=3)

mean_pt = (T_c.mean(), jak_rh.mean())
i_hot  = int(np.argmax(T_c))          # hottest hour
i_mug  = int(np.argmax(jak_rh))       # muggiest hour
i_best = int(np.nanargmax(jak_ecop))  # best-eCOP hour
i_worst= int(np.nanargmin(jak_ecop))  # worst-eCOP hour

def tag(ax, i, text, color, dx=0.4, dy=3):
    ax.scatter([T_c[i]], [jak_rh[i]], s=45, color=color, edgecolors="white",
               linewidths=0.8, zorder=6)
    ax.annotate(text, (T_c[i], jak_rh[i]), xytext=(T_c[i]+dx, jak_rh[i]+dy),
                fontsize=8, color=color, fontweight="bold", zorder=7,
                arrowprops=dict(arrowstyle="->", color=color, lw=0.8))

ax.scatter([mean_pt[0]], [mean_pt[1]], marker="*", s=260, color="#ffb703",
           edgecolors="#333", linewidths=0.8, zorder=6)
ax.annotate(f"year mean\n{mean_pt[0]:.1f}°C / {mean_pt[1]:.0f}%\neCOP {np.nanmean(jak_ecop):.1f}",
            mean_pt, xytext=(mean_pt[0]-5.6, mean_pt[1]-14), fontsize=8,
            fontweight="bold", color="#9a6a00", zorder=7,
            arrowprops=dict(arrowstyle="->", color="#9a6a00", lw=0.9))
tag(ax, i_hot,  f"hottest {T_c[i_hot]:.0f}°C\neCOP {jak_ecop[i_hot]:.1f}", "#d62828", dx=0.3, dy=-9)
tag(ax, i_mug,  f"muggiest {jak_rh[i_mug]:.0f}%\neCOP {jak_ecop[i_mug]:.1f}", "#023e8a", dx=-4.5, dy=2)
tag(ax, i_worst,f"worst eCOP {jak_ecop[i_worst]:.1f}", "#6a040f", dx=0.3, dy=6)
ax.set_title("3 — labelled: the story in a few points", fontsize=10, fontweight="bold")

fig.suptitle(f"Jakarta {year} on the eCOP map — a whole tropical year, hour by hour",
             fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print(f"Jakarta {year} sat almost entirely in the hot-humid corner: mean eCOP "
      f"{np.nanmean(jak_ecop):.2f}, and it NEVER left the top of the RH axis — "
      f"min RH all year was {jak_rh.min():.0f}%. The economizer has nowhere to breathe.")
